# Stage 2b: the race situation of every lap

This notebook takes the lap table and the pit-stop table of `02a_clean_laps.ipynb` and adds the race situation: track status, restart laps, weather and wet races for every lap, the pit loss of every race, and the flags `clean_lap` and `strategy_stop` that later stages filter on. It also builds the races table. `02c_positions.ipynb` follows.

Principle: *flag, don't delete*. No lap and no stop is removed here; they only get flags, and the flags are counted in the cleaning table.

Input: `data/processed/laps_02a.parquet` and `data/processed/stops_02a.parquet` (from `02a_clean_laps.ipynb`); race control, weather and pit files in `data/raw/`. Output: `data/processed/laps_02b.parquet`, `data/processed/stops_02b.parquet`, `data/processed/races.parquet`, `data/processed/cleaning_02b.csv` (this notebook's rows of the cleaning table; `02c_positions.ipynb` combines the rows of all three notebooks).

In [1]:
import time

import numpy as np
import pandas as pd

from helpers import (DATA_PROCESSED, KEY, LAP_KEY, add_step, find_red_flags, load_races, load_raw,
                     scheduled_start_times)

start_time = time.time()

# A race is wet if any car used one of these compounds.
WET_COMPOUNDS = ["INTERMEDIATE", "WET"]
# Pit loss: the driver's typical lap is the median of up to this many normal
# green laps on each side of the stop.
TYPICAL_LAP_WINDOW = 3
# A race with fewer usable stops than this takes the median of its circuit.
MIN_STOPS_FOR_PIT_LOSS = 3
# A stop whose pit-lane time is more than this many seconds above the median
# pit-lane time of its race (a slow stop or a served penalty) is left out of
# the pit loss estimate.
SLOW_STOP_LANE_TIME = 5
# The same for the six 2023 races without pit data, measured on the lap-based
# pit loss against the median pit loss of the race.
SLOW_STOP_PIT_LOSS = 8
# A stop that loses more than this many seconds above the pit loss of its race
# is not a strategy stop (likely repairs). Penalties of 5 or 10 s stay below it.
SLOW_STOP_FOR_STRATEGY = 20
# A lap slower than this multiple of the driver's median lap in the same stint
# is an outlier for the pace model.
OUTLIER_RATIO = 1.07

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 200)

# The rows this notebook adds to the cleaning table: one row per step.
cleaning_steps = []

## 1. Load the tables of 02a and the raw files

The lap table and the pit-stop table come from `02a_clean_laps.ipynb`. Race control and weather are read from the raw files with `load_raw` (in `helpers.py`). From the raw pit files only the list of races that have pit data is needed.

In [2]:
laps = pd.read_parquet(DATA_PROCESSED / "laps_02a.parquet")
pit_stops = pd.read_parquet(DATA_PROCESSED / "stops_02a.parquet")
races = load_races()

race_control = load_raw("race_control", columns=["session_key", "date", "category", "message"],
                        date_columns=["date"])
weather = load_raw(
    "weather",
    columns=["session_key", "date", "track_temperature", "air_temperature", "rainfall"],
    number_columns=["track_temperature", "air_temperature", "rainfall"],
    date_columns=["date"],
)
races_with_pit = set(load_raw("pit", columns=["session_key"])["session_key"])

scheduled_start = scheduled_start_times(races)
red_flags = find_red_flags(race_control)
# Both are the same for every lap of a race (set in 02a): the winner's number
# of laps, and the number of laps missing at the source.
winner_laps = laps.groupby("session_key")["race_laps"].first()
race_missing = laps.groupby("session_key")["race_missing_laps"].first()

print(f"races: {len(races)} | with pit data: {len(races_with_pit)} | red-flag periods: {len(red_flags)}")
for name, table in [("laps", laps), ("stops", pit_stops), ("race_control", race_control), ("weather", weather)]:
    print(f"{name}: {len(table):,} rows, {table['session_key'].nunique()} races")

races: 91 | with pit data: 85 | red-flag periods: 13
laps: 101,930 rows, 91 races
stops: 3,418 rows, 91 races
race_control: 10,164 rows, 91 races
weather: 14,600 rows, 91 races


## 2. Safety-car and virtual-safety-car periods

Race control announces a safety car with `SAFETY CAR DEPLOYED` and a virtual safety car (VSC) with `VIRTUAL SAFETY CAR DEPLOYED` (`VSC DEPLOYED` in 2026). The messages `SAFETY CAR IN THIS LAP` and `VSC ENDING` only announce the end. The period really ends with the next `TRACK CLEAR`, or earlier when the race is red-flagged, the chequered flag falls, or a VSC is upgraded to a safety car.

A second `SAFETY CAR DEPLOYED` while a safety car is already out (2026 Silverstone) continues the same period.

In [3]:
DEPLOYED = {"SAFETY CAR DEPLOYED": "safety_car", "VIRTUAL SAFETY CAR DEPLOYED": "vsc", "VSC DEPLOYED": "vsc"}
PERIOD_ENDS = {"TRACK CLEAR": "track clear", "SESSION ABORTED": "red flag", "CHEQUERED FLAG": "chequered flag"}

# The race control file of 2026 Mexico City also holds messages from the day
# before, so only messages from the scheduled race start on are used.
during_race = race_control["date"] >= race_control["session_key"].map(scheduled_start)
messages = race_control[during_race & race_control["message"].isin(list(DEPLOYED) + list(PERIOD_ENDS))]
messages = messages.sort_values(["session_key", "date"], kind="stable")

periods = []
for session_key, race_messages in messages.groupby("session_key"):
    current = None
    for message in race_messages.itertuples():
        kind = DEPLOYED.get(message.message)
        ended_by = PERIOD_ENDS.get(message.message)
        if kind == "safety_car" and current is not None and current["kind"] == "vsc":
            ended_by = "safety car"  # a VSC upgraded to a safety car
        if current is not None and ended_by is not None:
            current["end"] = message.date
            current["ended_by"] = ended_by
            periods.append(current)
            current = None
        if kind is not None and current is None:
            current = {"session_key": session_key, "kind": kind, "start": message.date}
    if current is not None:
        # No end message at all: the period runs to the last race control message.
        last_message = race_control.loc[race_control["session_key"] == session_key, "date"].max()
        current["end"] = last_message
        current["ended_by"] = "end of messages"
        periods.append(current)
periods = pd.DataFrame(periods).merge(races[["session_key", "year", "circuit"]], on="session_key")
periods["seconds"] = (periods["end"] - periods["start"]).dt.total_seconds()

print("deployment messages:", int(messages["message"].isin(list(DEPLOYED)).sum()), "| periods:", len(periods))
print("\nperiods by kind and by what ended them:")
print(periods.groupby(["kind", "ended_by"]).size().unstack(fill_value=0).to_string())
print("\nperiods per season:")
print(periods.groupby(["year", "kind"]).size().unstack(fill_value=0).to_string())
print("\nlength of the periods in seconds:")
print(periods.groupby("kind")["seconds"].describe().round(0).to_string())

deployment messages: 119 | periods: 118

periods by kind and by what ended them:
ended_by    chequered flag  red flag  safety car  track clear
kind                                                         
safety_car               2         8           0           51
vsc                      2         1           4           50

periods per season:
kind  safety_car  vsc
year                 
2023          16   10
2024          13    9
2025          17   13
2026          15   25

length of the periods in seconds:
            count   mean    std   min    25%    50%    75%     max
kind                                                              
safety_car   61.0  491.0  220.0  54.0  372.0  466.0  611.0  1036.0
vsc          57.0  134.0   75.0  20.0   79.0  117.0  185.0   325.0


## 3. Track status and restart laps

`track_status` of a lap is `red`, `safety_car`, `vsc` or `green`. A lap is marked when **any part of the car's own lap** (from its start to its end) overlaps a period, so cars that are not on the leader's lap are handled correctly. When a lap overlaps more than one kind of period, red wins over safety car, and safety car over VSC. The red-flag laps are the ones found in `02a_clean_laps.ipynb` (`red_flag_lap`).

`restart_lap` is the first green lap of a car directly after a safety-car lap or a red-flag lap: the field is bunched up (or starts from the grid). It is not set after a VSC, where the gaps are kept and the pace is normal straight away.

A few laps have no start time (lap 1 in some races) or no end time (retirement laps). They are judged on the part of the lap that is known; the 19 laps with no time at all (lap 1 of 2025 Miami) stay `green`.

Race control has no message for the end of the laps that are driven behind the safety car when a race is resumed after a red flag (or started behind the safety car). Those laps therefore stay `green` here; the first of them is the `restart_lap`, and the outlier rule in section 8 keeps the slow ones out of the pace model.

In [4]:
def overlaps_any(table, periods_table, start_column, end_column):
    """Return True for rows whose time window overlaps one of the periods of the same race."""
    result = pd.Series(False, index=table.index)
    for period in periods_table.itertuples():
        result |= ((table["session_key"] == period.session_key)
                   & (table[start_column] <= period.end)
                   & (table[end_column] >= period.start))
    return result


# The window of a lap. A missing start (lap 1 in a few races) is rebuilt from
# the end and the lap time, and when the lap time is missing too the lap counts
# as its end only. A lap without an end (retirement) counts as its start only.
rebuilt_start = laps["lap_end"] - pd.to_timedelta(laps["lap_duration"], unit="s")
laps["window_start"] = laps["date_start"].fillna(rebuilt_start).fillna(laps["lap_end"])
laps["window_end"] = laps["lap_end"].fillna(laps["date_start"])
print("laps without a start time:", int(laps["date_start"].isna().sum()),
      "| start rebuilt from end and lap time:", int((laps["date_start"].isna() & rebuilt_start.notna()).sum()),
      "| only the end is known:", int((laps["date_start"].isna() & rebuilt_start.isna() & laps["lap_end"].notna()).sum()),
      "| no time at all:", int(laps["window_start"].isna().sum()))
print("laps without an end time (only the start is known):", int((laps["lap_end"].isna() & laps["date_start"].notna()).sum()))
print()

laps["under_safety_car"] = overlaps_any(laps, periods[periods["kind"] == "safety_car"], "window_start", "window_end")
laps["under_vsc"] = overlaps_any(laps, periods[periods["kind"] == "vsc"], "window_start", "window_end")
laps["track_status"] = np.select([laps["red_flag_lap"], laps["under_safety_car"], laps["under_vsc"]],
                                 ["red", "safety_car", "vsc"], default="green")

laps = laps.sort_values(LAP_KEY).reset_index(drop=True)
by_car = laps.groupby(KEY)
previous_status = by_car["track_status"].shift(1)
follows_directly = by_car["lap_number"].shift(1) == laps["lap_number"] - 1
laps["restart_lap"] = ((laps["track_status"] == "green") & previous_status.isin(["safety_car", "red"])
                       & follows_directly)

# The first two green laps of a car after a red flag: the lap behind the safety
# car or extra formation lap, and the standing or rolling start. Race control
# does not mark them, and section 7 does not count stops on them as strategy stops.
status_two_back = by_car["track_status"].shift(2)
two_back_follows = by_car["lap_number"].shift(2) == laps["lap_number"] - 2
first_after_red = (laps["track_status"] == "green") & (previous_status == "red") & follows_directly
second_after_red = ((laps["track_status"] == "green") & (previous_status == "green") & follows_directly
                    & (status_two_back == "red") & two_back_follows)
laps["after_red_flag"] = first_after_red | second_after_red
print("first two green laps after a red flag:", int(laps["after_red_flag"].sum()),
      "| first:", int(first_after_red.sum()), "| second:", int(second_after_red.sum()))
print()

print("laps by track status:")
print(laps.groupby(["year", "track_status"]).size().unstack(fill_value=0).assign(
    total=lambda table: table.sum(axis=1)).to_string())
print("\nlaps touching both a safety car and a VSC (marked safety_car):", int((laps["under_safety_car"] & laps["under_vsc"]).sum()))
print("red-flag laps that also touch a safety car or VSC (marked red):",
      int((laps["red_flag_lap"] & (laps["under_safety_car"] | laps["under_vsc"])).sum()))
print("\nrestart laps:", int(laps["restart_lap"].sum()),
      "| after a safety car:", int((laps["restart_lap"] & (previous_status == "safety_car")).sum()),
      "| after a red flag:", int((laps["restart_lap"] & (previous_status == "red")).sum()))

add_step(cleaning_steps, "laps under a safety car", len(laps), (laps["track_status"] == "safety_car").sum(),
         "flagged track_status = safety_car",
         f"any part of the car's lap overlaps one of the {int((periods['kind'] == 'safety_car').sum())} safety-car periods")
add_step(cleaning_steps, "laps under a virtual safety car", len(laps), (laps["track_status"] == "vsc").sum(),
         "flagged track_status = vsc",
         f"any part of the car's lap overlaps one of the {int((periods['kind'] == 'vsc').sum())} VSC periods")
add_step(cleaning_steps, "restart laps", len(laps), laps["restart_lap"].sum(), "flagged restart_lap",
         "first green lap of a car directly after a safety-car or red-flag lap")

laps without a start time: 52 | start rebuilt from end and lap time: 0 | only the end is known: 33 | no time at all: 19
laps without an end time (only the start is known): 242



first two green laps after a red flag: 423 | first: 223 | second: 200

laps by track status:
track_status  green  red  safety_car  vsc  total
year                                            
2023          22979  160         989  311  24439
2024          25316   86         966  256  26624
2025          24278    0        1471  511  26260
2026          22201  149        1286  971  24607

laps touching both a safety car and a VSC (marked safety_car): 72
red-flag laps that also touch a safety car or VSC (marked red): 159

restart laps: 1061 | after a safety car: 838 | after a red flag: 223


How each period maps to laps: the number of cars and laps it touches and the first and last lap number among them. `one_block` says that for every car the marked laps are consecutive.

In [5]:
rows = []
for period in periods.itertuples():
    touched = laps[(laps["session_key"] == period.session_key) & (laps["window_start"] <= period.end)
                   & (laps["window_end"] >= period.start)]
    per_car = touched.groupby("driver_number")["lap_number"].agg(["min", "max", "size"])
    rows.append({"cars": len(per_car), "laps": len(touched), "first_lap": touched["lap_number"].min(),
                 "last_lap": touched["lap_number"].max(),
                 "one_block": bool((per_car["max"] - per_car["min"] + 1 == per_car["size"]).all())})
periods = pd.concat([periods.drop(columns=["cars", "laps", "first_lap", "last_lap", "one_block"], errors="ignore"),
                     pd.DataFrame(rows)], axis=1)
periods[["first_lap", "last_lap"]] = periods[["first_lap", "last_lap"]].astype("Int64")
shown = periods.assign(start=periods["start"].dt.strftime("%H:%M:%S"), end=periods["end"].dt.strftime("%H:%M:%S"))
print(shown[["year", "circuit", "kind", "start", "end", "ended_by", "cars", "laps", "first_lap", "last_lap",
             "one_block"]].to_string(index=False))

 year            circuit       kind    start      end       ended_by  cars  laps  first_lap  last_lap  one_block
 2023             Jeddah safety_car 17:31:10 17:38:27    track clear    19    71         17        20       True
 2023          Melbourne safety_car 05:04:27 05:09:12    track clear    19    57          1         3       True
 2023          Melbourne safety_car 05:14:24 05:17:02       red flag    18    36          7         8       True
 2023          Melbourne        vsc 05:48:36 05:50:52    track clear    17    34         18        19       True
 2023          Melbourne safety_car 06:38:32 06:41:07       red flag    16    34         53        55       True
 2023             Sakhir        vsc 16:10:19 16:12:08    track clear    18    31         40        42       True
 2023               Baku safety_car 11:22:24 11:29:44    track clear    19    69         10        13       True
 2023           Montreal        vsc 18:12:28 18:13:14    track clear    19    30          7     

## 4. Weather per lap

The weather file has one record per minute. Each lap takes the record nearest in time to the start of the lap (to its end when the start is unknown): track temperature and air temperature (degrees Celsius) and `rainfall` (0 or 1).

In [6]:
weather_columns = ["track_temperature", "air_temperature", "rainfall"]
laps = laps.drop(columns=weather_columns + ["weather_date"], errors="ignore")
weather_records = weather.rename(columns={"date": "weather_date"}).sort_values("weather_date")

# merge_asof needs sorted, non-missing times; laps without any time keep empty weather.
has_time = laps["window_start"].notna()
with_time = laps.loc[has_time, LAP_KEY + ["window_start"]].sort_values("window_start")
nearest = pd.merge_asof(with_time, weather_records, left_on="window_start", right_on="weather_date",
                        by="session_key", direction="nearest")
laps = laps.merge(nearest[LAP_KEY + weather_columns + ["weather_date"]], on=LAP_KEY, how="left",
                  validate="one_to_one")

distance = (laps["window_start"] - laps["weather_date"]).dt.total_seconds().abs()
print("weather records:", len(weather), "| races with weather:", weather["session_key"].nunique(), "of", len(races))
print("seconds between the lap start and the nearest weather record:")
print(distance.describe().round(1).to_string())
no_weather = laps["track_temperature"].isna()
print("\nlaps without weather (no start and no end time):", int(no_weather.sum()))
print(laps[no_weather].groupby(["year", "circuit", "lap_number"]).size().to_string())
print("\nlaps starting with rainfall = 1:", int((laps["rainfall"] == 1).sum()))
add_step(cleaning_steps, "laps without weather", len(laps), no_weather.sum(), "weather left empty",
         "the lap has no start and no end time, so no nearest weather record")

weather records: 14600 | races with weather: 91 of 91
seconds between the lap start and the nearest weather record:
count    101911.0
mean         15.0
std           8.7
min           0.0
25%           7.5
50%          14.9
75%          22.5
max          58.9

laps without weather (no start and no end time): 19
year  circuit  lap_number
2025  Miami    1             19

laps starting with rainfall = 1: 2863


## 5. Wet races

A race is wet if any car used intermediate or wet tyres on any lap. Wet races are excluded from the pit-strategy analysis. The flag is copied to every lap and every stop of the race.

The `rainfall` field is shown next to it for comparison only: it does not decide anything, because a track can be wet without rain falling and rain can fall without anyone needing wet tyres.

In [7]:
on_wet_tyres = laps["compound"].isin(WET_COMPOUNDS)
wet_sessions = set(laps.loc[on_wet_tyres, "session_key"])
laps["wet_race"] = laps["session_key"].isin(wet_sessions)
pit_stops["wet_race"] = pit_stops["session_key"].isin(wet_sessions)

wet_summary = laps[laps["wet_race"]].assign(on_wet_tyres=on_wet_tyres).groupby(["year", "circuit"]).agg(
    laps=("lap_number", "size"), wet_tyre_laps=("on_wet_tyres", "sum"), laps_with_rainfall=("rainfall", "sum"))
wet_summary["wet_tyre_share"] = (wet_summary["wet_tyre_laps"] / wet_summary["laps"]).round(3)
wet_summary["laps_with_rainfall"] = wet_summary["laps_with_rainfall"].astype(int)
print("wet races:", len(wet_sessions), "| dry races:", len(races) - len(wet_sessions))
print(wet_summary.to_string())

rain_laps = laps[~laps["wet_race"]].groupby(["year", "circuit"])["rainfall"].sum()
print("\ndry races with rainfall = 1 at the start of at least one lap (number of such laps):")
print(rain_laps[rain_laps > 0].astype(int).to_string())
add_step(cleaning_steps, "laps in wet races", len(laps), laps["wet_race"].sum(), "flagged wet_race",
         f"{len(wet_sessions)} races in which a car used intermediate or wet tyres")

wet races: 11 | dry races: 80
                        laps  wet_tyre_laps  laps_with_rainfall  wet_tyre_share
year circuit                                                                   
2023 Monte Carlo        1515            442                 293           0.292
     Zandvoort          1344            327                 289           0.243
2024 Interlagos         1136           1134                 833           0.998
     Montreal           1272            844                 324           0.664
     Silverstone         961            234                 411           0.243
2025 Melbourne           926            681                 292           0.735
     Silverstone         826            607                  98           0.735
     Spa-Francorchamps   879             20                   0           0.023
2026 Interlagos         1436            514                  11           0.358
     Kuala Lumpur       1145             20                   0           0.017
     Montr

## 6. Pit loss

**Per stop.** The time lost by stopping is the in-lap time plus the out-lap time, minus the driver's typical lap before the stop and the typical lap after it. The typical lap is the median of up to 3 *normal green laps* in the 3 laps before the in-lap (or after the out-lap): green track status, not an in-lap or out-lap, not lap 1, with a lap time. Summing in-lap and out-lap handles Monte Carlo, Melbourne and Montreal, where the stop sits inside the in-lap. A stop without a normal green lap on one of the two sides gets no pit loss.

**Per race.** The median over the *usable stops* of the race. A usable stop has a tyre change, is not `compound_uncertain`, is not during a red flag, has a green in-lap and a green out-lap, is not on lap 1, and is not a slow stop.

A slow stop is a problem at the stop or a served penalty, which says nothing about the normal cost of stopping. It is found within its own race, because the pit loss of a circuit can change between seasons (Singapore from 2025, Shanghai in 2026):
- in the 85 races with pit data, a stop whose pit-lane time is more than 5 s above the median pit-lane time of the race;
- in the six 2023 races without pit data, a stop whose lap-based pit loss is more than 8 s above the median pit loss of the race.

Both medians are taken over the stops of the race that meet the other conditions. The rule applies to all races, wet ones included. A stop without a pit-lane time in a race with pit data cannot be judged and stays.

A race with fewer than 3 usable stops takes the median over the usable stops at the same circuit in the dry races of all seasons (`pit_loss_source` = `circuit`).

The pit-lane time of the pit data (`lane_duration`, 85 races) is kept as a comparison. It is the time between the pit entry and exit lines. It is not the time lost: a car that stays out also needs time for that stretch of track, while braking before the entry line, accelerating after the exit line and the out-lap on cold tyres are not in it.

In [8]:
# In-lap and out-lap of every stop: track status, lap time and whether the car retired on it.
pit_stops = pit_stops.drop(columns=["in_lap_status", "in_lap_time", "in_lap_incomplete", "in_lap_after_red_flag",
                                    "out_lap_status", "out_lap_time", "out_lap_incomplete",
                                    "out_lap_after_red_flag"], errors="ignore")
lap_facts = laps[LAP_KEY + ["track_status", "lap_duration", "incomplete_lap", "after_red_flag"]]
for side in ["in_lap", "out_lap"]:
    facts = lap_facts.rename(columns={"lap_number": side, "track_status": f"{side}_status",
                                      "lap_duration": f"{side}_time", "incomplete_lap": f"{side}_incomplete",
                                      "after_red_flag": f"{side}_after_red_flag"})
    pit_stops = pit_stops.merge(facts, on=KEY + [side], how="left", validate="many_to_one")

normal_green = ((laps["track_status"] == "green") & ~laps["is_in_lap"] & ~laps["is_out_lap"]
                & (laps["lap_number"] > 1) & laps["lap_duration"].notna())
normal_laps = laps.loc[normal_green, LAP_KEY + ["lap_duration"]]


def typical_lap(stop_table, lap_column, offsets):
    """Return the median time of the driver's normal green laps at the given distances from a stop lap."""
    times = {}
    for offset in offsets:
        wanted = stop_table[KEY].assign(lap_number=stop_table[lap_column] + offset)
        times[offset] = wanted.merge(normal_laps, on=LAP_KEY, how="left")["lap_duration"].to_numpy()
    return pd.DataFrame(times, index=stop_table.index).median(axis=1)


window = range(1, TYPICAL_LAP_WINDOW + 1)
pit_stops["typical_lap_before"] = typical_lap(pit_stops, "in_lap", [-offset for offset in window])
pit_stops["typical_lap_after"] = typical_lap(pit_stops, "out_lap", list(window))
pit_stops["pit_loss"] = (pit_stops["in_lap_time"] + pit_stops["out_lap_time"]
                         - pit_stops["typical_lap_before"] - pit_stops["typical_lap_after"])

pit_stops["tyre_change"] = pit_stops["has_stint_change"] & ~pit_stops["same_set"].fillna(False).astype(bool)
candidate = (pit_stops["tyre_change"] & ~pit_stops["compound_uncertain"] & ~pit_stops["red_flag"]
             & (pit_stops["in_lap_status"] == "green") & (pit_stops["out_lap_status"] == "green")
             & (pit_stops["in_lap"] > 1))
with_loss = candidate & pit_stops["pit_loss"].notna()

# Slow stops, judged against the other stops of the same race: on the pit-lane
# time where the race has pit data, on the lap-based pit loss in the six races without.
race_medians = pit_stops[with_loss].groupby("session_key")[["lane_duration", "pit_loss"]].median()
above_lane_median = pit_stops["lane_duration"] - pit_stops["session_key"].map(race_medians["lane_duration"])
above_loss_median = pit_stops["pit_loss"] - pit_stops["session_key"].map(race_medians["pit_loss"])
has_pit_data = pit_stops["session_key"].isin(races_with_pit)
slow_by_lane_time = with_loss & has_pit_data & (above_lane_median > SLOW_STOP_LANE_TIME)
slow_by_pit_loss = with_loss & ~has_pit_data & (above_loss_median > SLOW_STOP_PIT_LOSS)
pit_stops["pit_loss_slow"] = slow_by_lane_time | slow_by_pit_loss
pit_stops["pit_loss_usable"] = with_loss & ~pit_stops["pit_loss_slow"]
usable = pit_stops[pit_stops["pit_loss_usable"]]
print("stops:", len(pit_stops), "| tyre change, green in-lap and out-lap, no red flag, not lap 1:", int(candidate.sum()),
      "| of which with a pit loss:", int(with_loss.sum()))
print(f"left out as slow stops: pit-lane time more than {SLOW_STOP_LANE_TIME} s above the race median:",
      int(slow_by_lane_time.sum()),
      f"| pit loss more than {SLOW_STOP_PIT_LOSS} s above the race median (six races without pit data):",
      int(slow_by_pit_loss.sum()))
print("slow stops left out in total:", int(pit_stops["pit_loss_slow"].sum()),
      "| in dry races:", int((pit_stops["pit_loss_slow"] & ~pit_stops["wet_race"]).sum()),
      "| in wet races:", int((pit_stops["pit_loss_slow"] & pit_stops["wet_race"]).sum()),
      "| usable stops:", len(usable))
print("slow stops left out, per season:", pit_stops[pit_stops["pit_loss_slow"]].groupby("year").size().to_dict())
print("stops with a pit loss but no pit-lane time in races with pit data (cannot be judged, kept):",
      int((with_loss & has_pit_data & pit_stops["lane_duration"].isna()).sum()))
print("candidates without a pit loss: no in-lap or out-lap time:",
      int((candidate & (pit_stops["in_lap_time"].isna() | pit_stops["out_lap_time"].isna())).sum()),
      "| no normal green lap before:", int((candidate & pit_stops["typical_lap_before"].isna()).sum()),
      "| no normal green lap after:", int((candidate & pit_stops["typical_lap_after"].isna()).sum()))

print("\nusable stops in dry races, seconds:")
dry_usable = usable[~usable["wet_race"]]
print(dry_usable[["pit_loss", "lane_duration", "stop_duration"]].describe().round(2).to_string())
both = dry_usable.dropna(subset=["pit_loss", "lane_duration"])
print("pit-lane time minus pit loss, per stop (seconds):")
print((both["lane_duration"] - both["pit_loss"]).describe().round(2).to_string())

stops: 3418 | tyre change, green in-lap and out-lap, no red flag, not lap 1: 2158 | of which with a pit loss: 2085
left out as slow stops: pit-lane time more than 5 s above the race median: 106 | pit loss more than 8 s above the race median (six races without pit data): 18
slow stops left out in total: 124 | in dry races: 95 | in wet races: 29 | usable stops: 1961
slow stops left out, per season: {2023: 37, 2024: 30, 2025: 29, 2026: 28}
stops with a pit loss but no pit-lane time in races with pit data (cannot be judged, kept): 2
candidates without a pit loss: no in-lap or out-lap time: 1 | no normal green lap before: 22 | no normal green lap after: 51

usable stops in dry races, seconds:
       pit_loss  lane_duration  stop_duration
count   1709.00        1638.00         780.00
mean      23.61          23.46           2.94
std       26.31           2.70           0.94
min      -24.20          17.00           1.80
25%       20.89          21.82           2.40
50%       22.44          23

In [9]:
per_race = usable.groupby("session_key").agg(pit_loss_stops=("pit_loss", "size"), own_pit_loss=("pit_loss", "median"),
                                             pit_lane_time=("lane_duration", "median"))
circuit_pit_loss = usable[~usable["wet_race"]].groupby("circuit")["pit_loss"].median()

race_pit_loss = races[["session_key", "year", "circuit"]].merge(per_race, on="session_key", how="left")
race_pit_loss["pit_loss_stops"] = race_pit_loss["pit_loss_stops"].fillna(0).astype(int)
enough_stops = race_pit_loss["pit_loss_stops"] >= MIN_STOPS_FOR_PIT_LOSS
race_pit_loss["pit_loss"] = race_pit_loss["own_pit_loss"].where(enough_stops,
                                                                race_pit_loss["circuit"].map(circuit_pit_loss))
race_pit_loss["pit_loss_source"] = np.where(enough_stops, "race", "circuit")

print("races with their own estimate:", int(enough_stops.sum()), "| with the circuit median:", int((~enough_stops).sum()))
print("\nraces that take the circuit median (own_pit_loss is what their own few stops would give):")
print(race_pit_loss.loc[~enough_stops, ["year", "circuit", "pit_loss_stops", "own_pit_loss", "pit_loss"]].round(2).to_string(index=False))
print("\nraces without a pit loss estimate:", int(race_pit_loss["pit_loss"].isna().sum()))

races with their own estimate: 85 | with the circuit median: 6

races that take the circuit median (own_pit_loss is what their own few stops would give):
 year    circuit  pit_loss_stops  own_pit_loss  pit_loss
 2023  Melbourne               2         21.45     20.87
 2024 Interlagos               1         25.29     22.24
 2025      Miami               1         18.07     20.01
 2026      Monza               2         31.76     25.30
 2026       Baku               2         25.50     21.54
 2026  Las Vegas               0           NaN     22.49

races without a pit loss estimate: 0


## 7. Strategy stops

Only strategy stops are used in the tests of H1 to H3. A stop is a strategy stop when **none** of the following holds; `not_strategy_reason` gives the first one that does, in this order:

1. `red_flag`: the stop is during a red flag;
2. `no_tyre_change`: no stint change, or the same set carries on (`same_set`, mostly safety-car passages through the pit lane);
3. `compound_uncertain`: the stop cannot be paired with a stint change;
4. `wet_race`: the race is wet;
5. `in_lap_not_green`: the in-lap is under a safety car or VSC (or has no lap row). Only the in-lap counts: when the track is green on the in-lap, the decision to stop was taken under green, even if a safety car comes out afterwards;
6. `lap_1`: the stop is at the end of lap 1 (damage, not strategy);
7. `restart_after_red_flag`: the in-lap is one of the first two green laps after a red flag, the lap behind the safety car or extra formation lap and the standing or rolling start (the same logic as lap 1);
8. `retirement`: the car retired on the in-lap or the out-lap, or has no out-lap;
9. `slow_stop`: the stop loses more than 20 s more than the pit loss of its race (likely repairs). A stop with a 5 or 10 s penalty stays. A stop without a pit loss (no normal green lap on one side) cannot be judged and stays.

In [10]:
race_loss = race_pit_loss.set_index("session_key")["pit_loss"]
pit_stops["above_race_pit_loss"] = pit_stops["pit_loss"] - pit_stops["session_key"].map(race_loss)
retirement = (pit_stops["out_lap_status"].isna() | pit_stops["out_lap_incomplete"].eq(True)
              | pit_stops["in_lap_incomplete"].eq(True))
stop_conditions = [
    ("red_flag", pit_stops["red_flag"]),
    ("no_tyre_change", ~pit_stops["tyre_change"]),
    ("compound_uncertain", pit_stops["compound_uncertain"]),
    ("wet_race", pit_stops["wet_race"]),
    ("in_lap_not_green", pit_stops["in_lap_status"] != "green"),
    ("lap_1", pit_stops["in_lap"] == 1),
    ("restart_after_red_flag", pit_stops["in_lap_after_red_flag"].eq(True)),
    ("retirement", retirement),
    ("slow_stop", pit_stops["above_race_pit_loss"] > SLOW_STOP_FOR_STRATEGY),
]
pit_stops["not_strategy_reason"] = np.select([condition for _, condition in stop_conditions],
                                             [name for name, _ in stop_conditions], default="")
pit_stops["strategy_stop"] = pit_stops["not_strategy_reason"] == ""

stop_reasons = pd.DataFrame({
    "stops_with_this_condition": [int(condition.sum()) for _, condition in stop_conditions],
    "first_reason_for": [int((pit_stops["not_strategy_reason"] == name).sum()) for name, _ in stop_conditions],
}, index=[name for name, _ in stop_conditions])
print("stops:", len(pit_stops), "| strategy stops:", int(pit_stops["strategy_stop"].sum()))
print(stop_reasons.to_string())
print("\nstrategy stops without a pit loss (cannot be judged as slow, kept):",
      int((pit_stops["strategy_stop"] & pit_stops["pit_loss"].isna()).sum()))
slow = pit_stops[pit_stops["not_strategy_reason"] == "slow_stop"].sort_values("above_race_pit_loss", ascending=False)
print("\nten largest slow_stop cases (seconds):")
print(slow.head(10)[["year", "circuit", "driver", "in_lap", "lane_duration", "stop_duration", "in_lap_time",
                     "out_lap_time", "pit_loss", "above_race_pit_loss"]].round(1).to_string(index=False))
print("\nstrategy stops between 5 and 20 s above the pit loss of their race (kept):",
      int((pit_stops["strategy_stop"] & pit_stops["above_race_pit_loss"].between(5, SLOW_STOP_FOR_STRATEGY)).sum()))
print("\nstrategy stops by the track status of the out-lap:")
print(pit_stops.loc[pit_stops["strategy_stop"], "out_lap_status"].value_counts().to_string())
print("\nstrategy stops per season:")
print(pit_stops.groupby("year").agg(stops=("strategy_stop", "size"), strategy_stops=("strategy_stop", "sum"),
                                    races_with_one=("session_key", lambda keys: keys[
                                        pit_stops.loc[keys.index, "strategy_stop"]].nunique())).to_string())
add_step(cleaning_steps, "pit stops that are not strategy stops (counts are stops, not laps)", len(pit_stops),
         (~pit_stops["strategy_stop"]).sum(), "flagged strategy_stop = False",
         "red flag, no tyre change, uncertain compound, wet race, in-lap not green, lap 1, first two laps after "
         "a red flag, retirement or slow stop; "
         "first reason in not_strategy_reason")

stops: 3418 | strategy stops: 1873
                        stops_with_this_condition  first_reason_for
red_flag                                      290               290
no_tyre_change                                337               265
compound_uncertain                             71                 0
wet_race                                      568               449
in_lap_not_green                             1113               492
lap_1                                         111                17
restart_after_red_flag                          1                 1
retirement                                      8                 1
slow_stop                                     423                30

strategy stops without a pit loss (cannot be judged as slow, kept): 76

ten largest slow_stop cases (seconds):
 year   circuit driver  in_lap  lane_duration  stop_duration  in_lap_time  out_lap_time  pit_loss  above_race_pit_loss
 2026 Melbourne    STR      34            NaN         

## 8. Clean laps for the pace model

A clean lap is a normal racing lap that shows the pace of car and tyre. A lap is clean when **none** of the following holds; `not_clean_reason` gives the first one that does, in this order: `lap_1`, `no_lap_time`, `red_flag`, `safety_car`, `vsc`, `restart`, `in_lap`, `out_lap`, `no_compound`, `compound_uncertain`, `outlier`.

**Outlier rule.** Among the laps that pass everything else, a lap is an outlier when it is slower than 1.07 times the driver's median lap in the same stint (the median over those remaining laps). There is no rule for fast laps. This removes laps with an incident, and the slow laps behind the safety car after a red flag that race control does not mark.

`clean_lap` does not depend on the weather: laps on wet tyres can be clean. Stage 3 uses `wet_race` to keep dry races only.

In [11]:
lap_conditions = [
    ("lap_1", laps["lap_number"] == 1),
    ("no_lap_time", laps["lap_duration"].isna()),
    ("red_flag", laps["track_status"] == "red"),
    ("safety_car", laps["track_status"] == "safety_car"),
    ("vsc", laps["track_status"] == "vsc"),
    ("restart", laps["restart_lap"]),
    ("in_lap", laps["is_in_lap"]),
    ("out_lap", laps["is_out_lap"]),
    ("no_compound", laps["compound"].isna()),
    ("compound_uncertain", laps["tyre_flag"] == "compound_uncertain"),
]
laps["not_clean_reason"] = np.select([condition for _, condition in lap_conditions],
                                     [name for name, _ in lap_conditions], default="")

# The driver's typical lap in the stint: the median over the laps that passed every condition above.
remaining = laps[laps["not_clean_reason"] == ""]
stint_median = remaining.groupby(KEY + ["stint_number"])["lap_duration"].transform("median")
ratio = remaining["lap_duration"] / stint_median
print("laps passing every condition before the outlier rule:", len(remaining))
print("lap time divided by the driver's median in the stint, quantiles:")
print(ratio.quantile([0.01, 0.05, 0.5, 0.95, 0.99, 0.999]).round(4).to_string())
for threshold in [1.05, OUTLIER_RATIO, 1.10]:
    print(f"slower than {threshold:.2f} x the stint median: {int((ratio > threshold).sum())} laps "
          f"({int((ratio[~remaining['wet_race']] > threshold).sum())} in dry races)")
laps.loc[ratio[ratio > OUTLIER_RATIO].index, "not_clean_reason"] = "outlier"
laps["clean_lap"] = laps["not_clean_reason"] == ""

reason_names = [name for name, _ in lap_conditions] + ["outlier"]
lap_reasons = pd.DataFrame({
    "laps_with_this_condition": [int(condition.sum()) for _, condition in lap_conditions]
                                + [int((laps["not_clean_reason"] == "outlier").sum())],
    "first_reason_for": [int((laps["not_clean_reason"] == name).sum()) for name in reason_names],
}, index=reason_names)
print(f"\nclean laps: {int(laps['clean_lap'].sum())} of {len(laps)} ({laps['clean_lap'].mean():.1%})")
print(lap_reasons.to_string())
assert not (laps["clean_lap"] & (laps["incomplete_lap"] | laps["beyond_classified"])).any()

add_step(cleaning_steps, "laps that are not clean for the pace model", len(laps), (~laps["clean_lap"]).sum(),
         "flagged clean_lap = False",
         "lap 1, no lap time, red flag, safety car, VSC, restart, in-lap, out-lap, no or uncertain compound, "
         "or outlier; first reason in not_clean_reason")
add_step(cleaning_steps, "of which outliers", len(laps), (laps["not_clean_reason"] == "outlier").sum(),
         "flagged not_clean_reason = outlier",
         f"slower than {OUTLIER_RATIO} times the driver's median lap in the same stint")

laps passing every condition before the outlier rule: 86512
lap time divided by the driver's median in the stint, quantiles:


0.010    0.9775
0.050    0.9898
0.500    1.0000
0.950    1.0181
0.990    1.0493
0.999    1.2897
slower than 1.05 x the stint median: 844 laps (275 in dry races)
slower than 1.07 x the stint median: 561 laps (191 in dry races)
slower than 1.10 x the stint median: 304 laps (133 in dry races)



clean laps: 85951 of 101930 (84.3%)
                    laps_with_this_condition  first_reason_for
lap_1                                   1839              1839
no_lap_time                              293               215
red_flag                                 395               358
safety_car                              4712              4482
vsc                                     2049              2013
restart                                 1061              1050
in_lap                                  3412              2272
out_lap                                 3414              2253
no_compound                              213                 1
compound_uncertain                      1130               935
outlier                                  561               561


## 9. Races table

One row per race. `safety_car_periods` and `vsc_periods` count the periods of section 2 (a VSC upgraded to a safety car counts once in each). `pit_loss` is the estimate of section 6 in seconds, `pit_loss_stops` the number of usable stops of the race (slow stops left out), and `pit_lane_time` the median pit-lane time of those stops where the pit data has it.

In [12]:
period_counts = periods.groupby(["session_key", "kind"]).size().unstack(fill_value=0)
race_table = races[["year", "session_key", "circuit", "date"]].rename(columns={"date": "scheduled_start"})
race_table["laps"] = race_table["session_key"].map(winner_laps).astype(int)
race_table["wet_race"] = race_table["session_key"].isin(wet_sessions)
race_table["safety_car_periods"] = race_table["session_key"].map(period_counts["safety_car"]).fillna(0).astype(int)
race_table["vsc_periods"] = race_table["session_key"].map(period_counts["vsc"]).fillna(0).astype(int)
race_table["red_flag"] = race_table["session_key"].isin(set(red_flags["session_key"]))
race_table["race_missing_laps"] = race_table["session_key"].map(race_missing).astype(int)
race_table = race_table.merge(
    race_pit_loss[["session_key", "pit_loss", "pit_loss_stops", "pit_loss_source", "pit_lane_time"]],
    on="session_key", how="left", validate="one_to_one")
race_table = race_table.sort_values(["year", "scheduled_start"]).drop(columns="scheduled_start").reset_index(drop=True)
print(race_table.round(2).to_string(index=False))

 year  session_key            circuit  laps  wet_race  safety_car_periods  vsc_periods  red_flag  race_missing_laps  pit_loss  pit_loss_stops pit_loss_source  pit_lane_time
 2023         7953             Sakhir    57     False                   0            1     False                  0     24.23              35            race            NaN
 2023         7779             Jeddah    50     False                   1            0     False                  0     20.23               7            race            NaN
 2023         7787          Melbourne    58     False                   3            1      True                  0     20.87               2         circuit            NaN
 2023         9070               Baku    51     False                   1            0     False                  0     22.26               6            race            NaN
 2023         9078              Miami    57     False                   0            0     False                  0     20.18          

## 10. Checks

Every safety-car and VSC period is one block of consecutive laps for every car, shown for three races; every race has a pit loss estimate and the estimates are plausible; flags and reasons agree.

In [13]:
# 1. Each period maps to a block of consecutive marked laps for every car.
print("1. periods:", len(periods), "| one block of consecutive laps for every car:", int(periods["one_block"].sum()),
      "| periods touching no lap:", int((periods["laps"] == 0).sum()))
print(periods.loc[periods["laps"] == 0, ["year", "circuit", "kind", "start", "end"]].to_string(index=False))
assert periods["one_block"].all()
# Every safety-car or VSC lap belongs to a period, and every period with laps has marked laps.
assert (laps["track_status"].isin(["safety_car", "vsc"])
        == ((laps["under_safety_car"] | laps["under_vsc"]) & ~laps["red_flag_lap"])).all()

EXAMPLE_RACES = [(2023, "Jeddah"), (2024, "Miami"), (2026, "Baku")]
for year, circuit in EXAMPLE_RACES:
    race_periods = periods[(periods["year"] == year) & (periods["circuit"] == circuit)]
    race_laps = laps[(laps["year"] == year) & (laps["circuit"] == circuit)]
    print(f"\n{year} {circuit}")
    print(race_periods.assign(start=race_periods["start"].dt.strftime("%H:%M:%S"),
                              end=race_periods["end"].dt.strftime("%H:%M:%S"))[
        ["kind", "start", "end", "ended_by", "first_lap", "last_lap"]].to_string(index=False))
    around = race_laps[race_laps["lap_number"].between(race_periods["first_lap"].min() - 2,
                                                       race_periods["last_lap"].max() + 2)]
    by_lap = around.groupby("lap_number").agg(
        leader_start=("date_start", "min"), cars=("driver_number", "size"),
        green=("track_status", lambda status: (status == "green").sum()),
        safety_car=("track_status", lambda status: (status == "safety_car").sum()),
        vsc=("track_status", lambda status: (status == "vsc").sum()),
        restart=("restart_lap", "sum"), median_lap_time=("lap_duration", "median"))
    by_lap["leader_start"] = by_lap["leader_start"].dt.strftime("%H:%M:%S")
    print(by_lap.round(1).to_string())

# 2. Every race has a pit loss estimate, and the estimates are plausible.
assert race_table["pit_loss"].notna().all()
print(f"\n2. races with a pit loss estimate: {int(race_table['pit_loss'].notna().sum())} of {len(race_table)}")
print(f"   range: {race_table['pit_loss'].min():.1f} to {race_table['pit_loss'].max():.1f} s"
      f" | dry races: {race_table.loc[~race_table['wet_race'], 'pit_loss'].min():.1f} to"
      f" {race_table.loc[~race_table['wet_race'], 'pit_loss'].max():.1f} s")
by_circuit = race_table[~race_table["wet_race"]].groupby("circuit").agg(
    dry_races=("pit_loss", "size"), pit_loss=("pit_loss", "median"), lowest=("pit_loss", "min"),
    highest=("pit_loss", "max"), pit_lane_time=("pit_lane_time", "median")).sort_values("pit_loss")
print("   five circuits with the lowest pit loss (median over their dry races, seconds):")
print(by_circuit.head(5).round(1).to_string())
print("   five circuits with the highest pit loss:")
print(by_circuit.tail(5).round(1).to_string())
print("   five lowest and five highest single races:")
ordered = race_table.sort_values("pit_loss")[["year", "circuit", "wet_race", "pit_loss", "pit_loss_stops",
                                              "pit_loss_source", "pit_lane_time"]]
print(pd.concat([ordered.head(5), ordered.tail(5)]).round(1).to_string(index=False))

# 3. Flags and reasons agree.
assert (laps["clean_lap"] == (laps["not_clean_reason"] == "")).all()
assert (pit_stops["strategy_stop"] == (pit_stops["not_strategy_reason"] == "")).all()
assert not pit_stops.loc[pit_stops["strategy_stop"], "wet_race"].any()
print("\n3. clean laps:", int(laps["clean_lap"].sum()), "| strategy stops:", int(pit_stops["strategy_stop"].sum()))

1. periods: 118 | one block of consecutive laps for every car: 118 | periods touching no lap: 1
 year circuit kind                     start                       end
 2025   Miami  vsc 2025-05-04 20:06:03+00:00 2025-05-04 20:09:08+00:00

2023 Jeddah
      kind    start      end    ended_by  first_lap  last_lap
safety_car 17:31:10 17:38:27 track clear         17        20


           leader_start  cars  green  safety_car  vsc  restart  median_lap_time
lap_number                                                                     
15             17:25:58    20     20           0    0        0             96.0
16             17:27:32    20     20           0    0        0             96.1
17             17:29:06    20      6          14    0        0            107.7
18             17:30:40    19      0          19    0        0            136.2
19             17:32:48    19      0          19    0        0            134.7
20             17:35:58    19      0          19    0        0            150.6
21             17:38:39    19     19           0    0       19             98.4
22             17:40:13    19     19           0    0        0             96.6

2024 Miami
      kind    start      end    ended_by  first_lap  last_lap
       vsc 20:37:57 20:38:49 track clear         22        23
safety_car 20:47:06 20:56:01 track clear         28        32
  


3. clean laps: 85951 | strategy stops: 1873


## 11. Save

The three tables that `02c_positions.ipynb` loads and this notebook's rows of the cleaning table. `02c_positions.ipynb` puts the rows of 02a, 02b and 02c together into the cleaning table in `results/`.

In [14]:
lap_columns = ["year", "session_key", "circuit", "driver_number", "driver", "team", "lap_number", "lap_duration",
               "duration_sector_1", "duration_sector_2", "duration_sector_3", "i1_speed", "i2_speed", "st_speed",
               "date_start", "lap_end", "is_pit_out_lap", "race_laps", "compound", "stint_number", "tyre_age",
               "used_set", "is_in_lap", "is_out_lap", "tyre_flag", "stint_shifted", "incomplete_lap",
               "beyond_classified", "disqualified", "no_lap_time", "date_start_missing", "time_mismatch",
               "red_flag_lap", "race_missing_laps", "track_status", "restart_lap", "after_red_flag", "track_temperature",
               "air_temperature", "rainfall", "wet_race", "clean_lap", "not_clean_reason"]
laps = laps[lap_columns].sort_values(["year", "session_key", "driver_number", "lap_number"]).reset_index(drop=True)

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
laps.to_parquet(DATA_PROCESSED / "laps_02b.parquet", index=False)
pit_stops = pit_stops.drop(columns=["in_lap_incomplete", "out_lap_incomplete", "in_lap_after_red_flag",
                                    "out_lap_after_red_flag"])
pit_stops.to_parquet(DATA_PROCESSED / "stops_02b.parquet", index=False)
race_table.to_parquet(DATA_PROCESSED / "races.parquet", index=False)
own_steps = pd.DataFrame(cleaning_steps)
own_steps.to_csv(DATA_PROCESSED / "cleaning_02b.csv", index=False)
print("saved laps_02b.parquet:", laps.shape, "| stops_02b.parquet:", pit_stops.shape,
      "| races.parquet:", race_table.shape)
print(f"saved cleaning_02b.csv: {len(own_steps)} rows")

saved laps_02b.parquet: (101930, 43) | stops_02b.parquet: (3418, 37) | races.parquet: (91, 13)
saved cleaning_02b.csv: 8 rows


## 12. Report

In [15]:
pd.set_option("display.max_colwidth", 120)
print("CLEANING TABLE (rows of this notebook)")
print(own_steps.to_string(index=False))

print("\nRACE SITUATION")
season = race_table.groupby("year").agg(
    races=("session_key", "size"), wet_races=("wet_race", "sum"), dry_races=("wet_race", lambda wet: (~wet).sum()),
    safety_car_periods=("safety_car_periods", "sum"), races_with_safety_car=("safety_car_periods", lambda n: (n > 0).sum()),
    vsc_periods=("vsc_periods", "sum"), races_with_vsc=("vsc_periods", lambda n: (n > 0).sum()),
    races_with_red_flag=("red_flag", "sum"))
season.loc["all"] = season.sum()
print(season.to_string())
print("\nwet races:", ", ".join(f"{race.year} {race.circuit}" for race in race_table[race_table["wet_race"]].itertuples()))

print("\nlaps by track status, restart laps and clean laps per season:")
lap_season = laps.groupby("year").agg(
    laps=("lap_number", "size"), safety_car=("track_status", lambda status: (status == "safety_car").sum()),
    vsc=("track_status", lambda status: (status == "vsc").sum()),
    red=("track_status", lambda status: (status == "red").sum()), restart=("restart_lap", "sum"),
    clean_laps=("clean_lap", "sum"))
lap_season.loc["all"] = lap_season.sum()
lap_season["clean_share"] = (lap_season["clean_laps"] / lap_season["laps"]).round(4)
print(lap_season.to_string())
print("\nclean-lap share per season, dry races only:")
print(laps[~laps["wet_race"]].groupby("year")["clean_lap"].agg(["size", "sum", "mean"]).rename(
    columns={"size": "laps", "sum": "clean_laps", "mean": "clean_share"}).round(4).to_string())
print("\nfirst reason a lap is not clean:")
print(laps["not_clean_reason"].replace("", "(clean)").value_counts().to_string())

print("\nstrategy stops per season:")
stop_season = pit_stops.groupby("year").agg(stops=("strategy_stop", "size"), strategy_stops=("strategy_stop", "sum"))
stop_season.loc["all"] = stop_season.sum()
print(stop_season.to_string())
print("\nfirst reason a stop is not a strategy stop:")
print(pit_stops["not_strategy_reason"].replace("", "(strategy stop)").value_counts().to_string())

print("\npit loss by circuit and season (seconds; * = wet race, c = circuit median used):")
marked = race_table.assign(value=race_table["pit_loss"].round(1).astype(str)
                           + np.where(race_table["wet_race"], "*", "")
                           + np.where(race_table["pit_loss_source"] == "circuit", "c", ""))
pit_loss_table = marked.pivot(index="circuit", columns="year", values="value").fillna("")
pit_loss_table["dry_median"] = race_table[~race_table["wet_race"]].groupby("circuit")["pit_loss"].median().round(1)
pit_loss_table["pit_lane_time"] = race_table.groupby("circuit")["pit_lane_time"].median().round(1)
print(pit_loss_table.sort_values("dry_median").to_string())

print("\nmissing values in laps:")
print(laps.isna().sum()[lambda counts: counts > 0].to_string())
print("missing values in stops:")
print(pit_stops.isna().sum()[lambda counts: counts > 0].to_string())
print("missing values in races:")
print(race_table.isna().sum()[lambda counts: counts > 0].to_string())

print("\nEXAMPLE ROWS: laps")
print(laps.sample(3, random_state=42).T.to_string())
print("\nEXAMPLE ROWS: stops")
print(pit_stops.sample(3, random_state=42).T.to_string())
print("\nEXAMPLE ROWS: races")
print(race_table.sample(3, random_state=42).T.to_string())

print(f"\nRunning time: {time.time() - start_time:.1f} seconds")

CLEANING TABLE (rows of this notebook)
                                                              step  laps_before  laps_affected                             action                                                                                                                                                                         reason
                                           laps under a safety car       101930           4712  flagged track_status = safety_car                                                                                                            any part of the car's lap overlaps one of the 61 safety-car periods
                                   laps under a virtual safety car       101930           2049         flagged track_status = vsc                                                                                                                   any part of the car's lap overlaps one of the 57 VSC periods
                                              

lap_duration           293
duration_sector_1      661
duration_sector_2      234
duration_sector_3      392
i1_speed             15391
i2_speed              1141
st_speed              8377
date_start              52
lap_end                261
compound               213
stint_number           209
tyre_age               209
used_set               209
track_temperature       19
air_temperature         19
rainfall                19
missing values in stops:
pit_date                225
pit_duration            225
lane_duration           225
stop_duration          1953
same_set                 71
in_lap_status             6
in_lap_time              26
out_lap_status            4
out_lap_time              8
typical_lap_before      391
typical_lap_after       526
pit_loss                833
above_race_pit_loss     833
missing values in races:
pit_lane_time    7

EXAMPLE ROWS: laps
                                              50275                             100437                            2